# CropGuard Maize Disease Model: Business Intelligence and ML Workflow


The goal

A farmer takes a photo of a maize leaf, and the model says one of five things:

Healthy,
Maize Leaf Blight,
Maize Common Rust,
Maize Gray Leaf Spot,
Reject

## 1. Business understanding and BI success criteria (CRISP-DM)


Stakeholders.
 Farmers, researchers, administrators 





## 2. Data source, collection method, and governance

source - [PlantVillage on Hugging Face](https://huggingface.co/datasets/mohanty/PlantVillage)

The dataset card describes images captured under controlled conditions. It is suitable as a reproducible benchmark, but may not represent camera variation, backgrounds, varieties


 the dataset has
 54,306 images



In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train_model.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train_model.py").is_file(), "Open/run this notebook from the CropGuard repository."
os.chdir(PROJECT_ROOT)
DATA_DIR = Path.home() / ".cache" / "cropguard"

print("Project:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("TensorFlow GPUs:", len(tf.config.list_physical_devices("GPU")))
print("Image cache:", DATA_DIR)

In [ ]:
from train_model import CLASS_NAMES, PLANTVILLAGE_LABELS, _plant_target

label_rows = [
    ("Corn_(maize)___healthy", "Healthy"),
    ("Corn_(maize)___Northern_Leaf_Blight", "Maize Leaf Blight"),
    ("Corn_(maize)___Common_rust_", "Maize Common Rust"),
    ("Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Maize Gray Leaf Spot"),
]
label_table = pd.DataFrame(label_rows, columns=["PlantVillage source label", "CropGuard output"])
display(label_table)
print("Ordered model outputs:", CLASS_NAMES)
assert [name for _, name in label_rows] == CLASS_NAMES[:4]
assert _plant_target("Tomato___healthy") == CLASS_NAMES.index("Reject")
assert _plant_target("Corn_(maize)___healthy") == CLASS_NAMES.index("Healthy")

## 4. Collect, prepare, train, and evaluate

runs the same train_model.py script
The script downloads images and loads MobileNetV2
trains a new top layer then fine-tunes the top backbone layers, and saves everything

In [ ]:
EPOCHS = 8
FINE_TUNE_EPOCHS = 4
BATCH_SIZE = 32
MAX_PER_CLASS = 1500  # Set to 200 for a quicker development run; 0 means no cap.

command = [
    sys.executable,
    "train_model.py",
    "--epochs", str(EPOCHS),
    "--fine-tune-epochs", str(FINE_TUNE_EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--max-per-class", str(MAX_PER_CLASS),
    "--data-dir", str(DATA_DIR),
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 5. Business intelligence results

All counts and scores below are read from the report produced by the completed run. They are intentionally not invented or prefilled. The test set is held out from fitting and early stopping; only consult these test results once the model configuration is fixed.

In [ ]:
from app.config import Config

REPORT_PATH = Path(Config.MODEL_PATH).with_name("training_report.json")
with REPORT_PATH.open(encoding="utf-8") as report_file:
    report = json.load(report_file)

print("Source:", report["dataset_url"])
print("Source revision:", report["dataset_revision"])
print("License:", report["dataset_license"])
print("Run timestamp (UTC):", report["created_at_utc"])
print("Model setup:", report["architecture"], report["optimizer"], "lr=", report["learning_rate"])
print("Leaf-group leakage check:", report["leaf_group_leakage_check"])
print("Seed:", report["seed"], "Image size:", report["image_size"])
print("Requested epochs:", report["epochs_requested"], "Fine-tuning epochs:", report["fine_tune_epochs_requested"], "Batch size:", report["batch_size"])
print("Fine-tuning retained:", report["fine_tuning"]["selected_by_validation_loss"])

counts = pd.DataFrame(report["split_counts"]).reindex(CLASS_NAMES)
counts.index.name = "Class"
display(counts)
counts.plot(kind="bar", figsize=(12, 5), rot=25, title="Image counts by class and partition")
plt.ylabel("Images")
plt.tight_layout()
plt.show()

metrics = report["test_metrics"]
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro precision", "Macro recall", "Macro F1"],
    "Test result": [metrics["accuracy"], metrics["macro_precision"], metrics["macro_recall"], metrics["macro_f1_score"]],
})
display(summary.style.format({"Test result": "{:.4f}"}))
per_class = pd.DataFrame(metrics["per_class"]).T
display(per_class.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1_score": "{:.3f}"}))

In [ ]:
source_counts = pd.DataFrame(report["source_label_counts"]).fillna(0).astype(int)
source_counts.index.name = "Original PlantVillage class"
display(source_counts.sort_values("train", ascending=False))

source_root = DATA_DIR / "plantvillage-selected"
source_examples = {
    "Healthy": "Corn_(maize)___healthy",
    "Maize Leaf Blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Maize Common Rust": "Corn_(maize)___Common_rust_",
    "Maize Gray Leaf Spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
}
example_paths = {
    label: next((source_root / folder).glob("*"))
    for label, folder in source_examples.items()
}
reject_path = next(
    path
    for folder in sorted(source_root.iterdir())
    if folder.is_dir() and not folder.name.startswith("Corn_(maize)___")
    for path in folder.glob("*")
)
example_paths["Reject (non-maize source example)"] = reject_path
fig, axes = plt.subplots(1, len(example_paths), figsize=(16, 4))
for ax, (label, path) in zip(axes, example_paths.items()):
    ax.imshow(plt.imread(path))
    ax.set_title(label)
    ax.axis("off")
fig.suptitle("Example images after source filtering")
fig.tight_layout()
plt.show()

In [ ]:
confusion = pd.DataFrame(
    metrics["confusion_matrix"],
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
)
display(confusion)
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(confusion.values, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Held-out test confusion matrix")
for row in range(len(CLASS_NAMES)):
    for col in range(len(CLASS_NAMES)):
        ax.text(col, row, int(confusion.iloc[row, col]), ha="center", va="center")
fig.colorbar(image, ax=ax, label="Images")
fig.tight_layout()
plt.show()

In [ ]:
history = pd.DataFrame(report["training_history"])
display(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key in ("loss", "val_loss"):
    if key in history:
        axes[0].plot(history.index + 1, history[key], marker="o", label=key)
for key in ("accuracy", "val_accuracy"):
    if key in history:
        axes[1].plot(history.index + 1, history[key], marker="o", label=key)
axes[0].set_title("Training / validation loss")
axes[1].set_title("Training / validation accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
plt.show()
print("Saved report:", REPORT_PATH)
print("Model:", Config.MODEL_PATH)
print("Class map:", Config.CLASS_INDEX_PATH)

## 6. CropGuard integration verification

CropGuard and the trainer share the same 224x224 preprocessing function, MobileNetV2 input scaling, output ordering, model path, and JSON class map. This smoke check loads the saved model through the application's inference wrapper and runs one cached healthy-maize image through the app's preprocessing/inference path. It verifies artifact compatibility and latency only; it does not assert that a single sample was classified correctly.

In [ ]:
from app.ml import inference
from app.ml.preprocessing import preprocess_for_inference

with Path(Config.CLASS_INDEX_PATH).open(encoding="utf-8") as class_file:
    index_to_name = json.load(class_file)
ordered_names = [index_to_name[str(i)] for i in range(len(index_to_name))]
assert ordered_names == CLASS_NAMES, (ordered_names, CLASS_NAMES)
assert Path(Config.MODEL_PATH).is_file(), Config.MODEL_PATH

healthy_images = list((DATA_DIR / "plantvillage-selected" / "Corn_(maize)___healthy").glob("*"))
assert healthy_images, "No cached healthy maize image was found after training."
sample_path = healthy_images[0]
input_array = preprocess_for_inference(str(sample_path), Config.IMAGE_TARGET_SIZE)
predicted_name, confidence, latency_ms = inference.predict(
    input_array,
    str(sample_path),
    {"MODEL_PATH": Config.MODEL_PATH, "CLASS_INDEX_PATH": Config.CLASS_INDEX_PATH},
    ordered_names,
)
assert not inference.is_using_demo_model(), "CropGuard loaded its demo fallback instead of the trained model."
assert predicted_name in ordered_names
print({"sample": sample_path.name, "prediction": predicted_name, "confidence": round(confidence, 4), "latency_ms": latency_ms})
print("CropGuard model-loading and inference contract: PASS")

## 7. Deployment, database, and monitoring

The trainer writes the model to the `MODEL_PATH` configured in `.env`, the ordered output map to `CLASS_INDEX_PATH`, and this run's report beside the model as `training_report.json`. Verify the app uses the same environment and that each non-`Reject` class name matches a maize `DiseaseClass` row at the corresponding output index.

